# Day 3 — Second Judge (Llama-3.1-8B) + Mechanism Analysis

**Self-contained.** Fresh runtime, **T4 GPU**, run top to bottom.

Two jobs today:
1. **P(slot 1) mechanism analysis** on the existing Qwen data — no GPU, runs in seconds, may explain the antagonism.
2. **Second judge** (Llama-3.1-8B-Instruct) over the same 340 items → `raw_llama.jsonl`, then cross-judge comparison.

Everything reads from and writes back to `/content/drive/MyDrive/judge-compound-bias`.

**Write the paper while Cell 9 runs (~1 h).** The paper existing beats the paper being thorough.

In [ ]:
#@title Cell 1 — Install + mount Drive + restore previous results
!pip install -q transformers accelerate bitsandbytes statsmodels pandas numpy datasets

import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - switch runtime to T4!")

from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/judge-compound-bias"
!mkdir -p results
!cp -v {DRIVE}/raw_*.jsonl results/ 2>/dev/null || echo "(no prior jsonl found - check the Drive folder)"
!ls -la results/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 11.5 MB/s eta 0:00:00
CUDA: True | Tesla T4
Mounted at /content/drive
'/content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl' -> 'results/raw_qwen.jsonl'
total 892
drwxr-xr-x 2 root root   4096 Aug 24 20:17 .
drwxr-xr-x 1 root root   4096 Aug 24 20:17 ..
-rw------- 1 root root 903573 Aug 24 20:17 raw_qwen.jsonl


In [ ]:
#@title Cell 2 — Transformations (identical to day 2 — do not change)
import re

def apply_position(resp_a, resp_b, level):
    if level == 0:
        return resp_a, resp_b, True
    return resp_b, resp_a, False

def apply_formatting(text, level):
    if level == 0:
        return text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return text
    return "### Explanation\n" + "\n".join(f"- {s}" for s in sentences)

_FILLER_CLOSERS = [" This point is worth keeping in mind.", " That is the key idea here.",
                   " This is the essential reasoning."]

def apply_length(text, level):
    if level == 0:
        return text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return text
    return " ".join(f"{s}{_FILLER_CLOSERS[i % len(_FILLER_CLOSERS)]}"
                    for i, s in enumerate(sentences))

print("transforms ready")

transforms ready


---
## PART A — Mechanism analysis on existing Qwen data (no GPU, ~10 seconds)
Does the positional prior weaken as surface perturbations are added? If P(slot 1) slides toward 50%, that explains the antagonism.

In [ ]:
#@title Cell 3 — P(slot 1) by surface condition: the positional prior
import pandas as pd, numpy as np, json

df_q = pd.DataFrame([json.loads(l) for l in open("results/raw_qwen.jsonl")])
d = df_q[df_q.slot_verdict.isin(["1","2"])].copy()
d["slot1"] = (d.slot_verdict == "1").astype(int)
print("overall P(slot 1):", round(d.slot1.mean(), 3), " (50% = no positional prior)")

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; den = 1 + z*z/n
    c = (p + z*z/(2*n))/den
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return round((c-h)*100, 1), round((c+h)*100, 1)

for cat in ["close", "anchor2"]:
    s = d[d.category == cat]
    if not len(s): continue
    t = s.groupby(["formatting","length"], as_index=False).agg(k=("slot1","sum"), n=("slot1","size"))
    t["P_slot1_pct"] = (100*t.k/t.n).round(1)
    t["CI"] = [wilson(r.k, r.n) for r in t.itertuples()]
    print(f"\n=== P(slot 1) — {cat} — pooled over position (50% = content-tracking) ===")
    print(t[["formatting","length","P_slot1_pct","CI","n"]].to_string(index=False))

print("\n--- sanity: by position level (close) — a true slot prior looks similar at both ---")
c = d[d.category == "close"]
print(c.groupby(["position","formatting","length"], as_index=False)
       .agg(P_slot1=("slot1","mean"), n=("slot1","size")).round(3).to_string(index=False))

# READ: prior drops toward 50% as F/L switch on  -> mechanism found, goes in discussion.
#       prior flat near overall value            -> honest null, mechanism stays open.

overall P(slot 1): 0.646  (50% = no positional prior)

=== P(slot 1) — close — pooled over position (50% = content-tracking) ===
 formatting  length  P_slot1_pct           CI   n
          0       0         80.6 (77.0, 83.7) 520
          0       1         60.4 (56.0, 64.5) 507
          1       0         76.1 (72.2, 79.5) 518
          1       1         54.1 (49.8, 58.4) 512

=== P(slot 1) — anchor2 — pooled over position (50% = content-tracking) ===
 formatting  length  P_slot1_pct           CI  n
          0       0         67.5 (56.6, 76.8) 80
          0       1         48.8 (38.1, 59.5) 80
          1       0         66.2 (55.4, 75.7) 80
          1       1         43.8 (33.4, 54.7) 80

--- sanity: by position level (close) — a true slot prior looks similar at both ---
 position  formatting  length  P_slot1   n
        0           0       0    0.804 260
        0           0       1    0.579 254
        0           1       0    0.749 259
        0           1       1    0.535 256

---
## PART B — Second judge: Llama-3.1-8B-Instruct
Requires HF token + accepted license at huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
#@title Cell 4 — HF login (Colab secret: key icon in sidebar, name it HF_TOKEN)
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("logged in")

logged in


In [ ]:
#@title Cell 5 — Rebuild the SAME 340 items (close + anchor + strong anchors)
from datasets import load_dataset
import pandas as pd

ds = load_dataset("lmsys/mt_bench_human_judgments")
raw = ds["human"].to_pandas()
raw = raw[raw["turn"] == 1]

def _extract(row):
    try:
        return pd.Series({"question": row["conversation_a"][0]["content"],
                          "response_a": row["conversation_a"][1]["content"],
                          "response_b": row["conversation_b"][1]["content"]})
    except Exception:
        return pd.Series({"question": None, "response_a": None, "response_b": None})

raw = pd.concat([raw.reset_index(drop=True),
                 raw.apply(_extract, axis=1).reset_index(drop=True)], axis=1)
raw = raw.dropna(subset=["question","response_a","response_b"])
MAX_CHARS = 2200
raw = raw[(raw.response_a.str.len() < MAX_CHARS) & (raw.response_b.str.len() < MAX_CHARS)
          & (raw.question.str.len() < 600)]

def classify(group):
    votes = group["winner"].tolist()
    n_tie = sum(v == "tie" for v in votes)
    n_a = sum(v == "model_a" for v in votes)
    n_b = sum(v == "model_b" for v in votes)
    cat = "close" if (n_tie > 0 or (n_a > 0 and n_b > 0)) else "anchor"
    r = group.iloc[0]
    return pd.Series({"question": r["question"], "response_a": r["response_a"],
                      "response_b": r["response_b"], "category": cat, "n_votes": len(votes)})

pairs = (raw.groupby(["question_id","model_a","model_b"], as_index=False)
            .apply(classify, include_groups=False).reset_index(drop=True))

close  = pairs[pairs.category == "close"].head(260)
anchor = pairs[pairs.category == "anchor"].head(40)
final  = pd.concat([close, anchor]).reset_index(drop=True)
items = [{"item_id": f"mtb_{i}_{r.category}", "question": r.question,
          "response_a": r.response_a, "response_b": r.response_b, "category": r.category}
         for i, r in final.iterrows()]

strong_pool = pairs[(pairs.category == "anchor") & (pairs.n_votes >= 2)]
strong_anchor_items = [
    {"item_id": f"mtbA{i}_anchor2", "question": r.question, "response_a": r.response_a,
     "response_b": r.response_b, "category": "anchor2"}
    for i, r in strong_pool.head(40).reset_index(drop=True).iterrows()]

items_all = items + strong_anchor_items
print(f"items_all = {len(items_all)}  (expect 340)")

# CRITICAL: item_ids must match day 2 exactly, or cross-judge comparison breaks.
prev = set(pd.DataFrame([__import__("json").loads(l)
           for l in open("results/raw_qwen.jsonl")]).item_id.unique())
now = set(i["item_id"] for i in items_all)
print("item_id overlap with day-2 run:", len(prev & now), "/", len(prev))
assert len(prev & now) == len(prev), "MISMATCH - stop and tell Claude before running"
print("OK - same items")

README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/gpt4_pair-00000-of-00001-c0b431264a(…): reconstructing file:   0%|          |  0.00B /  650kB            

data/gpt4_pair-00000-of-00001-c0b431264a(…): downloading bytes:           |  0.00B            

data/human-00000-of-00001-25f49108187592(…): reconstructing file:   0%|          |  0.00B /  739kB            

data/human-00000-of-00001-25f49108187592(…): downloading bytes:           |  0.00B            

Generating gpt4_pair split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating human split:   0%|          | 0/3355 [00:00<?, ? examples/s]

items_all = 340  (expect 340)
item_id overlap with day-2 run: 340 / 340
OK - same items


In [ ]:
#@title Cell 6 — Load Llama-3.1-8B-Instruct (4-bit) + SMOKE TEST (~8 min)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
JUDGE_NAME = "llama3.1-8b"
PROMPT_VERSION = "v1"

_JUDGE_TEMPLATE = """You are an impartial evaluator. You will be shown a question and two candidate answers, labelled [Answer 1] and [Answer 2]. Decide which answer is better, judging only on correctness, relevance, and helpfulness. Ignore differences in length, formatting, and ordering.

Question:
{question}

[Answer 1]
{first}

[Answer 2]
{second}

Respond with EXACTLY one line in the form:
Verdict: 1
or
Verdict: 2
or
Verdict: tie"""

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)
model.eval()

def _generate(prompt, max_new_tokens=16):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _parse_verdict(raw):
    m = re.search(r"verdict\s*[:\-]?\s*\**\s*(1|2|tie)", raw, re.IGNORECASE)
    if not m:
        m = re.search(r"\b(1|2|tie)\b", raw, re.IGNORECASE)
    if not m:
        return "UNPARSED"
    v = m.group(1).lower()
    return "TIE" if v == "tie" else v

def judge_pair(question, first, second):
    raw = _generate(_JUDGE_TEMPLATE.format(question=question, first=first, second=second))
    return {"slot_verdict": _parse_verdict(raw), "raw": raw, "prompt_version": PROMPT_VERSION}

def slot_to_ab(slot_verdict, first_slot_is_a):
    if slot_verdict in ("TIE", "UNPARSED"):
        return slot_verdict
    if slot_verdict == "1":
        return "A" if first_slot_is_a else "B"
    return "B" if first_slot_is_a else "A"

# ---- SMOKE TEST: check the raw string. Llama preambles more than Qwen. ----
for q, a, b in [("Why does ice float on water?",
                 "Water expands when it freezes, lowering its density.",
                 "Ice is less dense than liquid water, so it floats."),
                ("What is 2+2?", "It is 4.", "The answer is four, computed by addition.")]:
    print(judge_pair(q, a, b))
# If either shows UNPARSED or a chatty raw string -> paste it to Claude before Cell 9.

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'slot_verdict': '2', 'raw': 'Verdict: 2', 'prompt_version': 'v1'}
{'slot_verdict': '1', 'raw': 'Verdict: 1', 'prompt_version': 'v1'}


In [ ]:
#@title Cell 7 — Factorial harness (identical logic to day 2)
import itertools, json, time
from pathlib import Path

INCLUDE_LENGTH = True

def _factor_levels():
    f = {"position": [0,1], "formatting": [0,1]}
    if INCLUDE_LENGTH: f["length"] = [0,1]
    return f

def _all_conditions():
    f = _factor_levels(); keys = list(f)
    for combo in itertools.product(*[f[k] for k in keys]):
        yield dict(zip(keys, combo))

def _build_condition(resp_a, resp_b, cond):
    a = apply_formatting(resp_a, cond.get("formatting", 0))
    b = apply_formatting(resp_b, cond.get("formatting", 0))
    a = apply_length(a, cond.get("length", 0))
    b = apply_length(b, cond.get("length", 0))
    return apply_position(a, b, cond.get("position", 0))

def run_item(item, judge_name):
    base = {k: 0 for k in _factor_levels()}
    first, second, fia = _build_condition(item["response_a"], item["response_b"], base)
    res = judge_pair(item["question"], first, second)
    baseline_ab = slot_to_ab(res["slot_verdict"], fia)
    for cond in _all_conditions():
        first, second, fia = _build_condition(item["response_a"], item["response_b"], cond)
        res = judge_pair(item["question"], first, second)
        v = slot_to_ab(res["slot_verdict"], fia)
        flipped = (baseline_ab in ("A","B")) and (v in ("A","B")) and v != baseline_ab
        yield {"item_id": item["item_id"], "judge": judge_name,
               "category": item.get("category","na"),
               "position": cond.get("position",0), "formatting": cond.get("formatting",0),
               "length": cond.get("length",0), "first_slot_is_a": fia,
               "slot_verdict": res["slot_verdict"], "verdict_ab": v,
               "baseline_ab": baseline_ab, "flipped": bool(flipped),
               "raw_judge_output": res["raw"], "prompt_version": res["prompt_version"],
               "temperature": 0.0, "timestamp": time.time()}

def run_experiment(items, judge_name, out_path, progress_every=10):
    out = Path(out_path); out.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out.exists():
        for line in open(out):
            try: done.add(json.loads(line)["item_id"])
            except Exception: pass
    if done: print(f"RESUMING: {len(done)} items already in {out_path}")
    todo = [i for i in items if i["item_id"] not in done]
    t0 = time.time()
    with open(out, "a") as f:
        for n, it in enumerate(todo, 1):
            for rec in run_item(it, judge_name):
                f.write(json.dumps(rec) + "\n")
            f.flush()
            if n % progress_every == 0:
                rate = (time.time()-t0)/n
                print(f"{n}/{len(todo)} | ~{rate:.0f}s/item | ~{(len(todo)-n)*rate/60:.0f} min left")
    print("run complete ->", out)
    return str(out)

print("harness ready | calls/item:", 1 + 2**len(_factor_levels()))

harness ready | calls/item: 9


In [ ]:
#@title Cell 8 — PILOT: 10 items (~2 min). Check UNPARSED before the full hour.
_ = run_experiment(items_all[:10], JUDGE_NAME, "results/raw_llama.jsonl")
import pandas as pd, json
_p = pd.DataFrame([json.loads(l) for l in open("results/raw_llama.jsonl")])
print("UNPARSED rate:", round((_p.slot_verdict=="UNPARSED").mean(), 3), " (want < 0.05)")
print("TIE rate:", round((_p.slot_verdict=="TIE").mean(), 3))
print("flip rate:", round(_p.flipped.mean(), 3))
print("\nsample raw outputs:"); print(_p.raw_judge_output.head(5).tolist())
# GATE: UNPARSED > 5% -> stop, paste the raw outputs to Claude, fix the parser.

10/10 | ~12s/item | ~0 min left
run complete -> results/raw_llama.jsonl
UNPARSED rate: 0.0  (want < 0.05)
TIE rate: 0.0
flip rate: 0.325

sample raw outputs:
['Verdict: 2', 'Verdict: 2', 'Verdict: 2', 'Verdict: 2', 'Verdict: 1']


In [ ]:
#@title Cell 9 — FULL RUN (~1 h, resumable). WRITE THE PAPER WHILE THIS RUNS.
path = run_experiment(items_all, JUDGE_NAME, "results/raw_llama.jsonl")

DRIVE = "/content/drive/MyDrive/judge-compound-bias"
!cp -v results/raw_llama.jsonl {DRIVE}/
print("saved to Drive")

RESUMING: 10 items already in results/raw_llama.jsonl
10/330 | ~12s/item | ~62 min left
20/330 | ~10s/item | ~54 min left
30/330 | ~10s/item | ~52 min left
40/330 | ~11s/item | ~51 min left
50/330 | ~10s/item | ~47 min left
60/330 | ~10s/item | ~45 min left
70/330 | ~9s/item | ~41 min left
80/330 | ~9s/item | ~38 min left
90/330 | ~9s/item | ~36 min left
100/330 | ~9s/item | ~34 min left
110/330 | ~9s/item | ~33 min left
120/330 | ~9s/item | ~32 min left
130/330 | ~9s/item | ~30 min left
140/330 | ~9s/item | ~29 min left
150/330 | ~9s/item | ~27 min left
160/330 | ~9s/item | ~26 min left
170/330 | ~9s/item | ~25 min left
180/330 | ~10s/item | ~24 min left
190/330 | ~10s/item | ~23 min left
200/330 | ~10s/item | ~21 min left
210/330 | ~10s/item | ~20 min left
220/330 | ~10s/item | ~19 min left
230/330 | ~10s/item | ~17 min left
240/330 | ~11s/item | ~16 min left
250/330 | ~11s/item | ~14 min left
260/330 | ~11s/item | ~12 min left
270/330 | ~11s/item | ~11 min left
280/330 | ~11s/item |

In [ ]:
#@title Cell 10 — Cross-judge analysis (flip rates, antagonism, positional prior)
import numpy as np, pandas as pd, json, glob

df = pd.concat([pd.DataFrame([json.loads(l) for l in open(f)])
                for f in sorted(glob.glob("results/raw_*.jsonl"))], ignore_index=True)
print(df.groupby(["judge","category"]).item_id.nunique(), "\n")

FACTORS = ["position","formatting","length"]
u = df[df.baseline_ab.isin(["A","B"]) & df.verdict_ab.isin(["A","B"])].copy()

def indep(*ps):
    q = 1.0
    for x in ps: q *= (1-x)
    return 1-q

def paired_bootstrap(d, cell, parts, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    piv = d.pivot_table(index="item_id", columns=FACTORS, values="flipped")
    idx = piv.index.values
    out = []
    for _ in range(n_boot):
        s = piv.loc[rng.choice(idx, size=len(idx), replace=True)]
        out.append(s[cell].mean() - indep(*[s[c].mean() for c in parts]))
    return np.percentile(out, [2.5, 97.5])

for judge in sorted(u.judge.unique()):
    for cat in ["close","anchor2"]:
        d = u[(u.judge==judge) & (u.category==cat)]
        if not len(d): continue
        r = lambda p,f,l: d[(d.position==p)&(d.formatting==f)&(d.length==l)].flipped.mean()
        pos, fmt, ln = r(1,0,0), r(0,1,0), r(0,0,1)
        print(f"\n--- {judge} | {cat} (n items = {d.item_id.nunique()}) ---")
        print(f"singles: position {pos*100:.1f}%  formatting {fmt*100:.1f}%  length {ln*100:.1f}%")
        for name, cell, parts in [
            ("pos x fmt", (1,1,0), [(1,0,0),(0,1,0)]),
            ("pos x len", (1,0,1), [(1,0,0),(0,0,1)]),
            ("fmt x len", (0,1,1), [(0,1,0),(0,0,1)]),
            ("all three", (1,1,1), [(1,0,0),(0,1,0),(0,0,1)])]:
            obs = r(*cell); pred = indep(*[r(*c) for c in parts])
            lo, hi = paired_bootstrap(d, cell, parts)
            print(f"  {name:10s} pred {pred*100:5.1f}%  obs {obs*100:5.1f}%  "
                  f"diff {(obs-pred)*100:+6.1f}  CI [{lo*100:+.1f}, {hi*100:+.1f}]")

print("\n=== POSITIONAL PRIOR P(slot 1) by judge x surface condition ===")
s = df[df.slot_verdict.isin(["1","2"])].copy()
s["slot1"] = (s.slot_verdict=="1").astype(int)
print(s[s.category=="close"].groupby(["judge","formatting","length"], as_index=False)
       .agg(P_slot1=("slot1","mean"), n=("slot1","size")).round(3).to_string(index=False))

judge        category
llama3.1-8b  anchor       40
             anchor2      40
             close       260
qwen2.5-7b   anchor       40
             anchor2      40
             close       260
Name: item_id, dtype: int64 


--- llama3.1-8b | close (n items = 258) ---
singles: position 37.6%  formatting 11.8%  length 26.0%
  pos x fmt  pred  44.9%  obs  35.3%  diff   -9.6  CI [-13.8, -5.4]
  pos x len  pred  53.8%  obs  32.9%  diff  -20.9  CI [-27.1, -14.7]
  fmt x len  pred  34.7%  obs  23.3%  diff  -11.4  CI [-14.9, -7.8]
  all three  pred  59.2%  obs  30.6%  diff  -28.6  CI [-35.0, -22.2]

--- llama3.1-8b | anchor2 (n items = 40) ---
singles: position 15.0%  formatting 7.5%  length 20.0%
  pos x fmt  pred  21.4%  obs   7.5%  diff  -13.9  CI [-28.0, -2.4]
  pos x len  pred  32.0%  obs   5.0%  diff  -27.0  CI [-44.6, -9.1]
  fmt x len  pred  26.0%  obs  17.5%  diff   -8.5  CI [-19.0, +1.2]
  all three  pred  37.1%  obs   2.5%  diff  -34.6  CI [-52.0, -18.4]

--- qwen2.5-7b | close (

In [ ]:
#@title Cell 11 — Save everything to Drive (run at the end of EVERY session)
DRIVE = "/content/drive/MyDrive/judge-compound-bias"
!cp -v results/*.jsonl {DRIVE}/
!ls -la {DRIVE}/

cp: cannot stat 'results/*.jsonl': No such file or directory
ls: cannot access '/content/drive/MyDrive/judge-compound-bias/': No such file or directory


---
### Gates
1. **Cell 6 smoke test** — raw output must contain a parseable verdict.
2. **Cell 8 pilot** — UNPARSED < 5% before committing the hour.
3. **Cell 9** — resumable; rerun Cells 1-7 then Cell 9 after any disconnect.

### Interpreting Cell 10
- **Antagonism replicates on Llama** → "holds across judge families" (stronger general claim).
- **Antagonism differs on Llama** → "interaction structure is judge-dependent" (more nuanced, arguably more interesting).

Both are publishable. Write the paper either way.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

drive = Path("/content/drive/MyDrive")


for p in drive.rglob("raw_qwen.jsonl"):
    print(p)

for p in drive.rglob("raw_llama.jsonl"):
    print(p)

Mounted at /content/drive
/content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl
/content/drive/MyDrive/judge-compound-bias/raw_llama.jsonl


In [ ]:
# Cell 1 — Load existing saved judge outputs
# NO MODEL LOADING / NO GPU / NO INFERENCE

import json
import numpy as np
import pandas as pd
from pathlib import Path

QWEN_PATH = Path("/content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl")
LLAMA_PATH = Path("/content/drive/MyDrive/judge-compound-bias/raw_llama.jsonl")
assert QWEN_PATH.exists(), f"Cannot find {QWEN_PATH}"
assert LLAMA_PATH.exists(), f"Cannot find {LLAMA_PATH}"

def load_jsonl(path):
    with open(path, "r") as f:
        return pd.DataFrame([json.loads(line) for line in f if line.strip()])

df_qwen = load_jsonl(QWEN_PATH)
df_llama = load_jsonl(LLAMA_PATH)

df = pd.concat([df_qwen, df_llama], ignore_index=True)

print("Qwen rows:", len(df_qwen))
print("Llama rows:", len(df_llama))
print("Total saved factorial rows:", len(df))

print("\nItems by judge/category:")
print(
    df.groupby(["judge", "category"])
      .item_id.nunique()
)

Qwen rows: 2720
Llama rows: 2720
Total saved factorial rows: 5440

Items by judge/category:
judge        category
llama3.1-8b  anchor       40
             anchor2      40
             close       260
qwen2.5-7b   anchor       40
             anchor2      40
             close       260
Name: item_id, dtype: int64


In [ ]:
# Cell 2 — Diagnose ties / unparsed outputs / baseline exclusions

print("=== SLOT VERDICT COUNTS ===")
print(
    df.groupby(["judge", "category", "slot_verdict"])
      .size()
      .unstack(fill_value=0)
)

print("\n=== BASELINE_AB COUNTS BY UNIQUE ITEM ===")

baseline_per_item = (
    df.groupby(["judge", "category", "item_id"], as_index=False)
      .first()[["judge", "category", "item_id", "baseline_ab"]]
)

print(
    baseline_per_item
      .groupby(["judge", "category", "baseline_ab"])
      .size()
      .unstack(fill_value=0)
)

print("\n=== NON-BINARY BASELINE ITEMS ===")

bad_baselines = baseline_per_item[
    ~baseline_per_item["baseline_ab"].isin(["A", "B"])
]

print(bad_baselines.to_string(index=False))

=== SLOT VERDICT COUNTS ===
slot_verdict             1    2  TIE
judge       category                
llama3.1-8b anchor     124  196    0
            anchor2    150  170    0
            close     1129  941   10
qwen2.5-7b  anchor     165  155    0
            anchor2    181  139    0
            close     1396  661   23

=== BASELINE_AB COUNTS BY UNIQUE ITEM ===
baseline_ab             A   B  TIE
judge       category              
llama3.1-8b anchor     19  21    0
            anchor2    25  15    0
            close     169  89    2
qwen2.5-7b  anchor     25  15    0
            anchor2    30  10    0
            close     209  51    0

=== NON-BINARY BASELINE ITEMS ===
      judge category      item_id baseline_ab
llama3.1-8b    close mtb_71_close         TIE
llama3.1-8b    close mtb_74_close         TIE


In [ ]:
# Cell 5 — Original OR null vs parity/XOR null

def indep_or(*ps):
    """
    Original null:
    probability that at least one independent flip-cause occurs.
    """
    q = 1.0
    for p in ps:
        q *= (1.0 - p)
    return 1.0 - q


def indep_xor(*ps):
    """
    Independent-toggle / parity null:
    final verdict differs from baseline iff an odd number
    of independent perturbations toggle it.
    """
    prod = 1.0
    for p in ps:
        prod *= (1.0 - 2.0 * p)
    return (1.0 - prod) / 2.0


print("Sanity check:")
print("OR :", indep_or(.6, .3))
print("XOR:", indep_xor(.6, .3))

Sanity check:
OR : 0.72
XOR: 0.54


In [21]:
# Cell 7 — Calculate isolated and compound flip rates

def flip_rate(d, position, formatting, length):
    x = d[
        (d["position"] == position) &
        (d["formatting"] == formatting) &
        (d["length"] == length)
    ]
    return x["flipped"].mean()


def get_rates(d):
    return {
        "control": flip_rate(d, 0, 0, 0),

        # isolated perturbations
        "P": flip_rate(d, 1, 0, 0),
        "F": flip_rate(d, 0, 1, 0),
        "L": flip_rate(d, 0, 0, 1),

        # compound perturbations
        "PF":  flip_rate(d, 1, 1, 0),
        "PL":  flip_rate(d, 1, 0, 1),
        "FL":  flip_rate(d, 0, 1, 1),
        "PFL": flip_rate(d, 1, 1, 1),
    }

In [22]:
# Cell 8 — Compare OR and XOR predictions

for judge in ["qwen2.5-7b", "llama3.1-8b"]:

    d = u[
        (u["judge"] == judge) &
        (u["category"] == "close")
    ].copy()

    r = get_rates(d)

    print("\n" + "=" * 75)
    print(judge, "| close | n =", d["item_id"].nunique())
    print("=" * 75)

    print(
        f"Isolated flip rates: "
        f"P={100*r['P']:.2f}%  "
        f"F={100*r['F']:.2f}%  "
        f"L={100*r['L']:.2f}%"
    )

    comparisons = [
        ("P × F", "PF", [r["P"], r["F"]]),
        ("P × L", "PL", [r["P"], r["L"]]),
        ("F × L", "FL", [r["F"], r["L"]]),
        ("All three", "PFL", [r["P"], r["F"], r["L"]]),
    ]

    print()
    print(
        f"{'Condition':<15}"
        f"{'Observed':>12}"
        f"{'OR pred':>12}"
        f"{'XOR pred':>12}"
        f"{'OR diff':>12}"
        f"{'XOR diff':>12}"
    )

    for name, obs_key, ps in comparisons:

        obs = r[obs_key]
        pred_or = indep_or(*ps)
        pred_xor = indep_xor(*ps)

        print(
            f"{name:<15}"
            f"{100*obs:>11.2f}%"
            f"{100*pred_or:>11.2f}%"
            f"{100*pred_xor:>11.2f}%"
            f"{100*(obs-pred_or):>+11.2f}"
            f"{100*(obs-pred_xor):>+11.2f}"
        )


qwen2.5-7b | close | n = 260
Isolated flip rates: P=66.54%  F=9.27%  L=22.83%

Condition          Observed     OR pred    XOR pred     OR diff    XOR diff
P × F                62.93%      69.64%      63.47%      -6.70      -0.54
P × L                50.59%      74.18%      58.99%     -23.59      -8.39
F × L                28.91%      29.99%      27.87%      -1.08      +1.04
All three            44.92%      76.57%      57.32%     -31.65     -12.40

llama3.1-8b | close | n = 258
Isolated flip rates: P=37.60%  F=11.76%  L=25.97%

Condition          Observed     OR pred    XOR pred     OR diff    XOR diff
P × F                35.29%      44.94%      40.52%      -9.64      -5.22
P × L                32.95%      53.80%      44.04%     -20.86     -11.09
F × L                23.26%      34.68%      31.62%     -11.42      -8.37
All three            30.62%      59.24%      45.44%     -28.62     -14.82


In [23]:
# Cell 9 — Check sample size in every factorial cell

condition_counts = (
    u[u["category"] == "close"]
    .groupby(
        ["judge", "position", "formatting", "length"]
    )
    .agg(
        n_rows=("item_id", "size"),
        n_items=("item_id", "nunique")
    )
    .reset_index()
)

print(condition_counts.to_string(index=False))

      judge  position  formatting  length  n_rows  n_items
llama3.1-8b         0           0       0     258      258
llama3.1-8b         0           0       1     258      258
llama3.1-8b         0           1       0     255      255
llama3.1-8b         0           1       1     258      258
llama3.1-8b         1           0       0     258      258
llama3.1-8b         1           0       1     258      258
llama3.1-8b         1           1       0     255      255
llama3.1-8b         1           1       1     258      258
 qwen2.5-7b         0           0       0     260      260
 qwen2.5-7b         0           0       1     254      254
 qwen2.5-7b         0           1       0     259      259
 qwen2.5-7b         0           1       1     256      256
 qwen2.5-7b         1           0       0     260      260
 qwen2.5-7b         1           0       1     253      253
 qwen2.5-7b         1           1       0     259      259
 qwen2.5-7b         1           1       1     256      2

In [24]:
# Cell 10 — Complete-case paired XOR analysis + bootstrap

def paired_xor_analysis(
    df,
    judge,
    category,
    compound_cell,
    single_cells,
    n_boot=2000,
    seed=42
):
    """
    Compare observed compound flip rate against the independent-toggle
    (XOR/parity) prediction.

    Only items with binary A/B judgments in the baseline, compound
    condition, and every required single-factor condition are retained.

    Bootstrap resamples ITEMS, preserving pairing across conditions.
    """

    d = df[
        (df["judge"] == judge) &
        (df["category"] == category) &
        (df["baseline_ab"].isin(["A", "B"])) &
        (df["verdict_ab"].isin(["A", "B"]))
    ].copy()

    # Pivot so each row = one item
    piv = d.pivot_table(
        index="item_id",
        columns=["position", "formatting", "length"],
        values="flipped",
        aggfunc="first"
    )

    required_cells = [compound_cell] + list(single_cells)

    # Retain exactly the same items for every quantity in this comparison
    complete = piv.dropna(subset=required_cells).copy()

    # Point estimates
    obs = complete[compound_cell].mean()

    single_rates = [
        complete[cell].mean()
        for cell in single_cells
    ]

    pred = indep_xor(*single_rates)
    diff = obs - pred

    # Paired item bootstrap
    rng = np.random.default_rng(seed)
    boot_diffs = []

    n = len(complete)

    for _ in range(n_boot):
        positions = rng.integers(0, n, size=n)
        sample = complete.iloc[positions]

        obs_b = sample[compound_cell].mean()

        ps_b = [
            sample[cell].mean()
            for cell in single_cells
        ]

        pred_b = indep_xor(*ps_b)

        boot_diffs.append(obs_b - pred_b)

    lo, hi = np.percentile(boot_diffs, [2.5, 97.5])

    return {
        "n": n,
        "single_rates": single_rates,
        "pred": pred,
        "obs": obs,
        "diff": diff,
        "ci_low": lo,
        "ci_high": hi,
    }

In [25]:
# Cell 11 — Final corrected Table 1

compound_specs = [
    (
        "Position × Formatting",
        (1, 1, 0),
        [(1, 0, 0), (0, 1, 0)]
    ),
    (
        "Position × Length",
        (1, 0, 1),
        [(1, 0, 0), (0, 0, 1)]
    ),
    (
        "Formatting × Length",
        (0, 1, 1),
        [(0, 1, 0), (0, 0, 1)]
    ),
    (
        "All three",
        (1, 1, 1),
        [(1, 0, 0), (0, 1, 0), (0, 0, 1)]
    ),
]

results = []

for judge in ["qwen2.5-7b", "llama3.1-8b"]:

    print("\n" + "=" * 90)
    print(judge)
    print("=" * 90)

    for name, compound_cell, single_cells in compound_specs:

        r = paired_xor_analysis(
            df=df,
            judge=judge,
            category="close",
            compound_cell=compound_cell,
            single_cells=single_cells,
            n_boot=2000,
            seed=42
        )

        results.append({
            "judge": judge,
            "condition": name,
            "n": r["n"],
            "predicted_pct": 100 * r["pred"],
            "observed_pct": 100 * r["obs"],
            "diff_pp": 100 * r["diff"],
            "ci_low_pp": 100 * r["ci_low"],
            "ci_high_pp": 100 * r["ci_high"],
        })

        print(
            f"{name:<25} "
            f"n={r['n']:3d} | "
            f"pred={100*r['pred']:5.1f}% | "
            f"obs={100*r['obs']:5.1f}% | "
            f"diff={100*r['diff']:+6.1f} pp | "
            f"95% CI [{100*r['ci_low']:+.1f}, "
            f"{100*r['ci_high']:+.1f}]"
        )

table_xor = pd.DataFrame(results)


qwen2.5-7b
Position × Formatting     n=259 | pred= 63.4% | obs= 62.9% | diff=  -0.4 pp | 95% CI [-4.3, +3.3]
Position × Length         n=253 | pred= 58.7% | obs= 50.6% | diff=  -8.1 pp | 95% CI [-13.2, -2.9]
Formatting × Length       n=253 | pred= 27.8% | obs= 29.2% | diff=  +1.4 pp | 95% CI [-2.6, +5.6]
All three                 n=253 | pred= 57.1% | obs= 44.3% | diff= -12.8 pp | 95% CI [-18.8, -7.0]

llama3.1-8b
Position × Formatting     n=255 | pred= 40.0% | obs= 35.3% | diff=  -4.7 pp | 95% CI [-8.4, -1.1]
Position × Length         n=258 | pred= 44.0% | obs= 32.9% | diff= -11.1 pp | 95% CI [-16.2, -6.3]
Formatting × Length       n=255 | pred= 31.9% | obs= 23.5% | diff=  -8.3 pp | 95% CI [-11.3, -5.4]
All three                 n=255 | pred= 45.2% | obs= 29.8% | diff= -15.4 pp | 95% CI [-20.3, -10.6]


In [28]:
# Cell 12 — Strong-consensus robustness analysis

anchor_results = []

for judge in ["qwen2.5-7b", "llama3.1-8b"]:

    print("\n" + "=" * 90)
    print(judge, "| strong consensus (anchor2)")
    print("=" * 90)

    for name, compound_cell, single_cells in compound_specs:

        r = paired_xor_analysis(
            df=df,
            judge=judge,
            category="anchor2",
            compound_cell=compound_cell,
            single_cells=single_cells,
            n_boot=2000,
            seed=42
        )

        anchor_results.append({
            "judge": judge,
            "condition": name,
            "n": r["n"],
            "predicted_pct": 100 * r["pred"],
            "observed_pct": 100 * r["obs"],
            "diff_pp": 100 * r["diff"],
            "ci_low_pp": 100 * r["ci_low"],
            "ci_high_pp": 100 * r["ci_high"],
        })

        print(
            f"{name:<25} "
            f"n={r['n']:3d} | "
            f"pred={100*r['pred']:5.1f}% | "
            f"obs={100*r['obs']:5.1f}% | "
            f"diff={100*r['diff']:+6.1f} pp | "
            f"95% CI [{100*r['ci_low']:+.1f}, "
            f"{100*r['ci_high']:+.1f}]"
        )

anchor_xor = pd.DataFrame(anchor_results)


qwen2.5-7b | strong consensus (anchor2)
Position × Formatting     n= 40 | pred= 35.8% | obs= 35.0% | diff=  -0.8 pp | 95% CI [-10.4, +10.0]
Position × Length         n= 40 | pred= 41.8% | obs= 20.0% | diff= -21.7 pp | 95% CI [-29.8, -13.7]
Formatting × Length       n= 40 | pred= 23.9% | obs= 42.5% | diff= +18.6 pp | 95% CI [+5.0, +34.0]
All three                 n= 40 | pred= 42.2% | obs= 20.0% | diff= -22.2 pp | 95% CI [-30.6, -13.2]

llama3.1-8b | strong consensus (anchor2)
Position × Formatting     n= 40 | pred= 20.2% | obs=  7.5% | diff= -12.8 pp | 95% CI [-24.4, -2.1]
Position × Length         n= 40 | pred= 29.0% | obs=  5.0% | diff= -24.0 pp | 95% CI [-38.0, -8.8]
Formatting × Length       n= 40 | pred= 24.5% | obs= 17.5% | diff=  -7.0 pp | 95% CI [-15.6, +2.2]
All three                 n= 40 | pred= 32.1% | obs=  2.5% | diff= -29.6 pp | 95% CI [-40.0, -16.9]


In [29]:
# Cell 13 — Save corrected results

from pathlib import Path

OUT = Path("/content/drive/MyDrive/judge-compound-bias/results")
OUT.mkdir(parents=True, exist_ok=True)

table_xor.to_csv(
    OUT / "xor_complete_case_close.csv",
    index=False
)

anchor_xor.to_csv(
    OUT / "xor_complete_case_anchor2.csv",
    index=False
)

print("Saved:")
print(OUT / "xor_complete_case_close.csv")
print(OUT / "xor_complete_case_anchor2.csv")

Saved:
/content/drive/MyDrive/judge-compound-bias/results/xor_complete_case_close.csv
/content/drive/MyDrive/judge-compound-bias/results/xor_complete_case_anchor2.csv
